# 04 · Evaluation & Visualisation

End-to-end validation of all three trained models:
1. LightGBM forecaster — MAE, RMSE, R² per horizon, actual vs predicted scatter
2. LSTM Autoencoder — anomaly score distribution, event windows highlighted
3. XGBoost classifier — confusion matrix, class-level precision/recall/F1
4. Mismatch heatmap (hour-of-day × month)

In [ ]:
import sys
sys.path.insert(0, '..')

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import (
    classification_report, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score,
)

from src.models import (
    MODELS_DIR, PROCESSED_DIR,
    LGB_FEATURE_COLS, XGB_FEATURE_COLS, LSTM_FEATURE_COLS,
    _load_parquet, _get_parquet_columns,
)

sns.set_theme(style='whitegrid', palette='tab10')
%matplotlib inline

## 1 · LightGBM Forecaster Evaluation

In [ ]:
# Load saved metrics
lgbm_metrics = joblib.load(MODELS_DIR / 'lgbm_metrics.pkl')

records = []
for key, m in lgbm_metrics.items():
    source, horizon = key.split('_')
    records.append({'source': source, 'horizon': horizon, **m})
df_lgbm = pd.DataFrame(records)
print(df_lgbm.to_string(index=False))

In [ ]:
# Bar chart: MAE by model
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, metric in zip(axes, ['mae', 'r2']):
    pivot = df_lgbm.pivot(index='horizon', columns='source', values=metric)
    pivot.plot(kind='bar', ax=ax, rot=0)
    ax.set_title(f'LightGBM {metric.upper()} by horizon')
    ax.set_xlabel('Forecast horizon')
    ax.set_ylabel('MAE (MW)' if metric == 'mae' else 'R²')

plt.tight_layout()
plt.show()

In [ ]:
# Actual vs predicted scatter for solar 24h model
model_path = MODELS_DIR / 'lgbm_solar_24h.txt'
if model_path.exists():
    model = lgb.Booster(model_file=str(model_path))
    need = LGB_FEATURE_COLS + ['solar_t24h_ahead']
    need = [c for c in dict.fromkeys(need) if c in _get_parquet_columns()]
    df_eval = _load_parquet(columns=need).dropna().sample(10000, random_state=42)
    y_true = df_eval['solar_t24h_ahead'].values
    y_pred = model.predict(df_eval[LGB_FEATURE_COLS].values.astype('float32'))

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(y_true, y_pred, alpha=0.3, s=5)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect')
    ax.set_xlabel('Actual (MW)')
    ax.set_ylabel('Predicted (MW)')
    ax.set_title(f'Solar 24h — R²={r2_score(y_true, y_pred):.4f}  MAE={mean_absolute_error(y_true, y_pred):.1f} MW')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 2 · LSTM Anomaly Score Analysis

In [ ]:
scores_path = MODELS_DIR / 'anomaly_scores.parquet'
if scores_path.exists():
    anom_df = pd.read_parquet(scores_path)
    anom_df['ts'] = pd.to_datetime(anom_df['ts_unix'], unit='s', utc=True)
    print(f'Rows: {len(anom_df):,}')
    print(anom_df.describe())

In [ ]:
threshold_info = joblib.load(MODELS_DIR / 'lstm_threshold.pkl') if (MODELS_DIR / 'lstm_threshold.pkl').exists() else {}
threshold = threshold_info.get('threshold', None)

if 'anom_df' in dir():
    # Plot anomaly scores over time for ERCO (Texas)
    erco = anom_df[anom_df['ba'] == 'ERCO'].sort_values('ts')

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.fill_between(erco['ts'], erco['anomaly_score'], alpha=0.5, label='Anomaly score')
    if threshold:
        ax.axhline(threshold, color='red', linestyle='--', label=f'99th pct ({threshold:.3f})')

    # 2021 Texas freeze window
    freeze_start = pd.Timestamp('2021-02-10', tz='UTC')
    freeze_end   = pd.Timestamp('2021-02-20', tz='UTC')
    ax.axvspan(freeze_start, freeze_end, alpha=0.2, color='orange', label='2021 TX Freeze')

    ax.set_title('ERCO Anomaly Scores — 2021')
    ax.set_xlabel('Date')
    ax.set_ylabel('Reconstruction MSE')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Anomaly score distribution
if 'anom_df' in dir() and threshold:
    scores = anom_df['anomaly_score'].dropna()
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.hist(scores, bins=100, log=True, color='steelblue', alpha=0.7)
    ax.axvline(threshold, color='red', linestyle='--', label=f'Threshold ({threshold:.3f})')
    ax.set_xlabel('Anomaly score (MSE)')
    ax.set_ylabel('Count (log)')
    ax.set_title('Anomaly Score Distribution — All BAs')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 3 · XGBoost Mismatch Classifier Evaluation

In [ ]:
xgb_model_path = MODELS_DIR / 'xgb_mismatch.json'
if xgb_model_path.exists():
    clf = xgb.XGBClassifier()
    clf.load_model(str(xgb_model_path))
    feature_names = joblib.load(MODELS_DIR / 'xgb_feature_names.pkl')

    need = [c for c in dict.fromkeys(feature_names + ['mismatch_label']) if c in _get_parquet_columns()]
    df_test = _load_parquet(columns=need).dropna(subset=['mismatch_label'])
    df_test = df_test.sample(min(50000, len(df_test)), random_state=99)

    available = [c for c in feature_names if c in df_test.columns]
    X_test = df_test[available].values.astype('float32')
    y_test = df_test['mismatch_label'].values.astype(int)
    y_pred = clf.predict(X_test)

    label_names = ['balanced', 'mod_surplus', 'sev_surplus', 'deficit']
    print(classification_report(y_test, y_pred, target_names=label_names))

In [ ]:
# Confusion matrix
if 'y_test' in dir():
    cm = confusion_matrix(y_test, y_pred, normalize='true')
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt='.2f', cmap='Blues',
        xticklabels=label_names, yticklabels=label_names, ax=ax
    )
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title('XGBoost Mismatch Classifier — Normalised Confusion Matrix')
    plt.tight_layout()
    plt.show()

In [ ]:
# Top 20 feature importances
if xgb_model_path.exists():
    importances = clf.feature_importances_
    feat_imp = pd.Series(importances, index=available).sort_values(ascending=False).head(20)

    fig, ax = plt.subplots(figsize=(8, 6))
    feat_imp.plot(kind='barh', ax=ax)
    ax.invert_yaxis()
    ax.set_xlabel('Importance (gain)')
    ax.set_title('XGBoost — Top 20 Feature Importances')
    plt.tight_layout()
    plt.show()

## 4 · Mismatch Heatmap — Hour-of-Day × Month

In [ ]:
heat_cols = ['hour_of_day', 'month', 'mismatch_label', 'ba']
heat_cols_avail = [c for c in heat_cols if c in _get_parquet_columns()]
heat_df = _load_parquet(columns=heat_cols_avail).dropna()

LABEL_NAMES = {0: 'Balanced', 1: 'Mod surplus', 2: 'Sev surplus', 3: 'Deficit'}
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, label in zip(axes.flat, [0, 1, 2, 3]):
    pivot = (
        heat_df.groupby(['hour_of_day', 'month'])
        .apply(lambda g: (g['mismatch_label'] == label).mean())
        .reset_index(name='proportion')
        .pivot(index='hour_of_day', columns='month', values='proportion')
    )
    sns.heatmap(
        pivot,
        ax=ax,
        cmap='RdYlGn_r' if label in (2, 3) else 'RdYlGn',
        cbar_kws={'label': 'Proportion'},
        vmin=0, vmax=pivot.max().max(),
    )
    ax.set_title(f'Class {label}: {LABEL_NAMES[label]}')
    ax.set_xlabel('Month')
    ax.set_ylabel('Hour of Day (UTC)')

plt.suptitle('Mismatch Severity Heatmap — Hour-of-Day × Month', fontsize=14)
plt.tight_layout()
plt.show()